# Exploratory Data Analysis

In any project of data analytics, we start with an important task called exploratory data analysis (EDA), i.e., using transformation, summarization, and visualisation to explore our data in a systematic way, often in an iterative cycle. 

In EDA, you:
- Ask some interesting questions about your data.
- Search for answers by visualising, transforming, and modeling your data.
- Use what you learn to refine your questions and/or generate new questions.

EDA is more than a formal process with a strict set of rules. Instead, think of EDA as a state of mind. Initially, feel free to explore different ideas. Some ideas may be dead ends while some may lead to key insights.

## Goal: to develop an understanding of your data. 

EDA is fundamentally a creative process. And like most creative processes, the key to asking quality questions is to generate a large amount of questions. Answering these questions can help you know what insights are contained in your dataset, expose you to new aspects, and increase your chance of making a discovery. 

There is no rule about which questions you should ask to guide your research. However, two types of questions will always be useful for making discoveries within your data. You can loosely word these questions as:

1. What type of variation occurs within my variables?
2. What type of covariation occurs between my variables?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
diamonds = sns.load_dataset('diamonds')
diamonds.head()

In [ ]:
diamonds.info()

In [ ]:
diamonds.describe(include='all')

## Variation

Variation is the tendency of the values of a variable to change from measurement to measurement. You can see variation easily in real life. The best way to understand that pattern is to visualize the distribution of the variable’s values.

### Visualizing Distribution 

How you visualise the distribution of a variable will depend on whether the variable is categorical or continuous. 

- Categorical variable: Use a bar chart.

In [ ]:
diamonds.cut.value_counts()

In [ ]:
diamonds.cut.value_counts().plot(kind='bar')
plt.show()

In [ ]:
sns.countplot(data=diamonds, x='cut')
plt.show()

- Continuous variable: Use a histogram. 

In [ ]:
# histograms for all numerical features
diamonds.hist(figsize=(15,15))
plt.show()

In [ ]:
diamonds['carat'].hist()
plt.show()

In [ ]:
# Specify the number of bins (default=10)
diamonds['carat'].hist(bins=20)
plt.show()

In [ ]:
# Specify the bin width
binwidth = 0.25
diamonds['carat'].hist(bins=np.arange(0, diamonds['carat'].max()+binwidth, binwidth))
plt.show()

In [ ]:
# To zoom in on diamonds of carat<=3
diamonds[diamonds.carat<=3]['carat'].hist(bins=20)
plt.show()

The Seaborn package has a variety of methods for visualizing distributions of data: 

https://seaborn.pydata.org/tutorial/distributions.html

In [ ]:
sns.histplot(diamonds, x='carat')

In [ ]:
sns.histplot(diamonds, x='carat', stat='probability')

### Typical values

In both bar charts and histograms, tall bars show the common values of a variable, and shorter bars show less-common values. Places that do not have bars reveal values that were not seen in your data. To turn this information into useful questions, look for anything unexpected:
- Which values are the most common? Why?
- Which values are rare? Why? Does that match your expectations?
- Can you see any unusual patterns? What might explain them?

### Unusual values

Outliers are observations that are unusual; data points that don’t seem to fit the pattern. Sometimes outliers are data entry errors; other times outliers suggest important new science. When you have a lot of data, outliers are sometimes difficult to see in a histogram. For example, take the distribution of the y variable from the diamonds dataset. 

In [ ]:
diamonds['y'].hist(bins=50)
plt.show()
# Look how wide the limits on the x-axis are. 

In [ ]:
# Let's zoom in to find where the outliers are. 
ax = diamonds['y'].hist(bins=50)
ax.set_ylim(ymin=0, ymax = 20)
plt.show()

In [ ]:
# The chart above allows us to see that there are three unusual values: 0, ~30, and ~60. 
# Let's pluck them out. 
diamonds[(diamonds.y<3)|(diamonds.y>20)][['x','y','z']].sort_values('y')

# The y variable measures one of the three dimensions of these diamonds, in mm. 
# Diamonds can’t have a width of 0mm, so these values must be incorrect. 
# Measurements of 32mm and 59mm are implausible. Over an inch long!!!

In [ ]:
# To drop the rows with the unusual values:
diamonds2 = diamonds[(diamonds.y>=3)&(diamonds.y<=20)]
print(f'Row count in the original dataset: {diamonds.shape[0]}')
print(f'Row count in the new dataset:      {diamonds2.shape[0]}')

## Covariation

If variation describes the behavior within a variable, covariation describes the behavior between variables. 

Covariation is the tendency for the values of two or more variables to vary together in a related way. The best way to spot covariation is to visualise the relationship between two or more variables. 

### A categorical and a continuous variable

It’s common to want to explore the distribution of a continuous variable broken down by a categorical variable. 

In [ ]:
# How the price of a diamond varies with its cut. 
sns.histplot(diamonds, x="price", hue="cut",bins=20) 

In [ ]:
# The histogram above is not easy to see the differences between different cuts. 
# We can change the argument multiple to make it better. 

sns.histplot(diamonds, x="price", hue="cut",bins=20,multiple='stack') 

#sns.histplot(diamonds, x="price", hue="cut",bins=20,multiple='dodge') 

In [ ]:
# You can use a displot() as well. 
sns.displot(diamonds, x="price", hue="cut",bins=20,multiple='stack') 

In [ ]:
# Set stat='probability' to change the scale of y axis
sns.displot(diamonds, x="price", hue="cut",bins=20,multiple='dodge',stat='probability') 

These histograms seem not that useful for comparison because the height is given by the count. That means if one of the groups is much smaller than the others, it’s hard to see the differences in shape, e.g., the "fair" cut in this dataset.

In [ ]:
# By setting common_norm=False, each subset will be normalized independently:
sns.displot(diamonds, x="price", hue="cut",bins=20,multiple='dodge',stat='probability',common_norm=False) 

### KDE plot
A histogram aims to approximate the underlying probability density function that generated the data by binning and counting observations. Kernel density estimation (KDE) presents a different solution to the same problem. Rather than using discrete bins, a KDE plot smooths the observations with a Gaussian kernel, producing a continuous density estimate: 

In [ ]:
# Use a KDE plot to visualize the distribution of a continuous variable
sns.displot(diamonds, x="price", kind="kde") 

In [ ]:
# You can use kdeplot() as well
sns.kdeplot(data=diamonds, x="price")

In [ ]:
# Use a KDE plot to visualize the distributions of price for different cuts
sns.displot(diamonds, x="price", hue="cut", kind="kde") 

In [ ]:
sns.kdeplot(data=diamonds, x="price", hue='cut')

Setting the argument ``common_norm``.
- If **common_norm=True**, scale each conditional density by the number of observations such that the total area under all densities sums to 1. 
- If **common_norm=False**, normalize each density independently such that the area under each density sums to 1.

In [ ]:
# Normalize density for groups: The area below each curve is 1. 

sns.kdeplot(data=diamonds, x="price", hue="cut", common_norm=False)
#sns.displot(diamonds, x="price", hue="cut", kind="kde", common_norm=False)

There’s something rather surprising about this plot - it appears that fair diamonds (the lowest quality) have the highest average price! 

### Boxplot

Another alternative to display the distribution of a continuous variable broken down by a categorical variable is the boxplot. A boxplot is a type of visual shorthand for a distribution of values that is popular among statisticians. Each boxplot consists of:

- A box that stretches from the 25th percentile of the distribution to the 75th percentile, a distance known as the interquartile range (IQR). In the middle of the box is a line that displays the median, i.e. 50th percentile, of the distribution. These three lines give you a sense of the spread of the distribution and whether or not the distribution is symmetric about the median or skewed to one side.
- Visual points that display observations that fall more than 1.5 times the IQR from either edge of the box. These outlying points are unusual so are plotted individually.
- A line (or whisker) that extends from each end of the box and goes to the farthest non-outlier point in the distribution.

In [ ]:
# how to display an image
from IPython.display import Image
Image('https://raw.githubusercontent.com/jiexunli-wwu/mis433/refs/heads/main/img/eda-boxplot.png')

In [ ]:
# We can use a boxplot to identify outliers for y. 
diamonds['y'].plot.box()

In [ ]:
# Create a boxplot on price
diamonds['price'].plot.box()
#diamonds['price'].plot(kind='box')

In [ ]:
# Distribution of price by cut
sns.boxplot(data=diamonds, x='cut', y='price')

Compared to histograms, we see much less information about the distribution, but the boxplots are much more compact so we can more easily compare them (and fit more on one plot). It supports the counterintuitive finding that better quality diamonds are cheaper on average! 

In [ ]:
my_order = diamonds.groupby('cut')['price'].median().sort_values().index
ax = sns.boxplot(data=diamonds, x='cut', y='price', order=my_order)
ax.set_yscale("log") # Transform the y scale to log for easier comparison

### Two continuous variables

If you want to visualise the covariation between two continuous variables, draw a scatterplot. 

In [ ]:
diamonds.plot(x='carat', y='price', kind='scatter')
plt.show()

In [ ]:
sns.scatterplot(data=diamonds,x='carat',y='price')

In [ ]:
# Adjust alpha to add transparency
diamonds.plot(x='carat', y='price', kind='scatter', alpha=0.1)
#sns.scatterplot(data=diamonds,x='carat',y='price',alpha=0.1)
plt.show()

### Two categorical variables

To visualise the covariation between categorical variables, you’ll need to count the number of observations for each combination. 

In [ ]:
dcc = diamonds.groupby(['cut','color'])['carat'].agg(['count'])
dcc

In [ ]:
# Simply using a scatterplot may not show the pattern. 
sns.scatterplot(data=diamonds, x='cut', y='color')

In [ ]:
# The size of each circle in the plot displays how many observations occurred at each combination of values. 
sns.scatterplot(data=dcc, x='cut', y='color', size='count')

In [ ]:
# Create pivot table. 
dpt = diamonds.pivot_table(values=['carat'], index=['cut'], columns=['color'], aggfunc=np.size)
dpt

In [ ]:
# Create a heatmap
sns.heatmap(data=dpt)